# Web application implementation and technical evaluation
Run after notebook 01. Generates the app from the selected, independently tested pipeline.
Free hosting: Streamlit Community Cloud. Entry point: `deployment/app.py`.

In [ ]:
%pip -q install numpy==2.2.6 pandas==2.2.3 scipy==1.15.3 scikit-learn==1.7.2 xgboost==3.0.5 shap==0.48.0 matplotlib==3.10.3 joblib==1.5.1

%pip -q install streamlit==1.50.0 pytest==8.4.2 requests==2.32.5

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Final Research - C D Jayasinghe')
for folder in ['data', 'models', 'outputs', 'report_figures', 'deployment', 'notebooks']:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

## 1. Generate deployment files

In [ ]:
import json, shutil, subprocess, sys, os
APP_ROOT = ROOT / 'deployment'
(APP_ROOT / 'models').mkdir(exist_ok=True)
(APP_ROOT / '.streamlit').mkdir(exist_ok=True)
for name in ['h3_pipeline.joblib','schema.json']:
    source = ROOT / 'models' / name
    if not source.is_file():
        raise FileNotFoundError('Run notebook 01 before creating the application.')
    shutil.copy2(source, APP_ROOT / 'models' / name)

In [ ]:
files = {'core.py': "\nfrom pathlib import Path\nimport json\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport shap\n\nBASE = Path(__file__).resolve().parent\n\ndef load_bundle():\n    model = joblib.load(BASE / 'models/h3_pipeline.joblib')\n    schema = json.loads((BASE / 'models/schema.json').read_text(encoding='utf-8'))\n    if list(model.classes_) != [0,1,2] or list(model.feature_names_in_) != schema['features']:\n        raise ValueError('Model and schema do not agree.')\n    return model, schema\n\ndef validate(values, features):\n    if not isinstance(values, dict) or set(values) != set(features):\n        raise ValueError('Answer all 20 questions.')\n    for value in values.values():\n        if isinstance(value, (bool, str)) or value is None:\n            raise ValueError('Select one response from 0 to 4 for every question.')\n        if not np.isfinite(value) or value != int(value) or not 0 <= value <= 4:\n            raise ValueError('Responses must be whole numbers from 0 to 4.')\n    return pd.DataFrame([[values[f] for f in features]], columns=features, dtype=float)\n\ndef estimate(model, schema, values):\n    frame = validate(values, schema['features'])\n    probability = model.predict_proba(frame)[0]\n    if len(probability) != 3 or not np.isfinite(probability).all() or not np.isclose(probability.sum(),1):\n        raise ValueError('Invalid model output.')\n    return frame, probability, int(probability.argmax())\n\ndef explain(model, schema, frame):\n    features = schema['features']\n    background = pd.DataFrame(np.repeat(np.arange(5)[:,None], len(features), axis=1), columns=features, dtype=float)\n    def predict(values):\n        return model.predict_proba(pd.DataFrame(values, columns=features))\n    explanation = shap.Explainer(predict, background, algorithm='permutation', seed=42)(frame, max_evals=2*len(features)+1)\n    np.testing.assert_allclose(explanation.base_values + explanation.values.sum(axis=1), model.predict_proba(frame), atol=1e-6)\n    return explanation\n\ndef guidance(class_id):\n    messages = [\n        'If professional support feels difficult to approach, you can first ask a university counselling service about confidentiality, cost, and appointment options. You decide whether and when to contact them.',\n        'If you are unsure about professional support, you can ask a university counselling service what a first appointment involves before deciding.',\n        'If you would like professional support, you can contact your university counselling service or a qualified mental-health professional to discuss appointment options.'\n    ]\n    return messages[class_id]\n", 'app.py': '\nimport pandas as pd\nimport streamlit as st\nfrom core import load_bundle, estimate, explain, guidance\n\nst.set_page_config(page_title=\'Help-Seeking Intention | Research DSS\', page_icon=\'🌿\', layout=\'centered\')\nst.markdown("""<style>\n.stApp { background: #f6f8f7; }\n.block-container { max-width: 900px; padding-top: 2rem; }\nh1,h2,h3 { color: #185e57; }\ndiv.stButton > button, div.stFormSubmitButton > button { border-radius: 9px; }\n</style>""", unsafe_allow_html=True)\nst.caption(\'SRI LANKAN UNIVERSITY STUDENT RESEARCH\')\nst.title(\'Understand help-seeking intentions\')\nst.write(\'An estimate of stated willingness to seek professional counselling, based on dating-violence questionnaire responses.\')\nst.link_button(\'Leave this page\', \'https://www.google.com\')\nst.caption(\'Leaving does not remove browser history. Responses are processed for this session and are not saved by this app.\')\n\n@st.cache_resource\ndef bundle():\n    return load_bundle()\n\ntry:\n    model, schema = bundle()\nexcept Exception:\n    st.error(\'The trained model is unavailable. Please try again later.\')\n    st.stop()\n\nwith st.expander(\'About this research\'):\n    st.write(schema[\'title\'])\n    st.write(\'H3 classes: low = 1–3, neutral = 4, high = 5–7. The app estimates intention from D01–D20; it does not ask for H3.\')\n    st.write(\'This estimates intention, not actual service use, diagnosis, or violence severity. It does not decide whether you need support.\')\n    st.write(f"Selected model: {schema[\'model\']}. Development responses: {schema[\'development_n\']}; independent test responses: {schema[\'test_n\']}.")\n    st.dataframe(pd.DataFrame([schema[\'test_metrics\']]).T.rename(columns={0:\'Independent test score\'}), use_container_width=True)\n    st.caption(\'Survey sample results do not establish performance for every university or student.\')\n\nst.info(\'You may stop at any time. Support remains an option whatever the estimated intention. If you feel unsafe now, contact local emergency services or someone you trust.\')\neligible = st.checkbox(\'I am aged 18 or over, am a Sri Lankan university student in a current dating relationship, and choose to continue.\', key=\'eligible\')\nwith st.form(\'questionnaire\'):\n    st.subheader(\'Your relationship experiences\')\n    st.caption(\'Use the same reference period as the study questionnaire. Select a frequency for every item; no response is preselected.\')\n    answers = {}\n    for feature in schema[\'features\']:\n        answers[feature] = st.selectbox(f"{feature}. {schema[\'questions\'][feature]}", options=list(range(5)),\n            index=None, placeholder=\'Choose a response\', format_func=lambda v: f"{v} — {schema[\'scale\'][str(v)]}", key=feature)\n    submitted = st.form_submit_button(\'Estimate intention\', type=\'primary\')\n\nif submitted:\n    if not eligible:\n        st.warning(\'Confirm eligibility and your choice to continue before estimating.\')\n    else:\n        try:\n            frame, probabilities, predicted = estimate(model, schema, answers)\n        except ValueError as error:\n            st.warning(str(error))\n        else:\n            st.subheader(schema[\'labels\'][predicted])\n            st.caption(\'Estimated category of stated professional help-seeking intention\')\n            table = pd.DataFrame({\'Category\': schema[\'labels\'], \'Estimated probability\': probabilities})\n            st.dataframe(table, hide_index=True, use_container_width=True)\n            st.bar_chart(table.set_index(\'Category\'))\n            st.write(guidance(predicted))\n            st.caption(\'Every category has uncertainty. A low intention estimate does not mean low violence, low need, or that support should be withheld.\')\n            with st.spinner(\'Explaining this estimate…\'):\n                try:\n                    explanation = explain(model, schema, frame)\n                    contributions = explanation.values[0,:,predicted]\n                    detail = pd.DataFrame({\'Question\':schema[\'features\'],\n                        \'Response\':[schema[\'scale\'][str(int(frame.iloc[0][f]))] for f in schema[\'features\']],\n                        \'Probability contribution\': contributions})\n                    order = detail[\'Probability contribution\'].abs().sort_values(ascending=False).index\n                    st.subheader(\'What influenced the estimate\')\n                    st.caption(\'Positive values increase this category’s probability relative to the reference; negative values decrease it. These are model associations, not causes.\')\n                    st.dataframe(detail.loc[order].head(8), hide_index=True, use_container_width=True)\n                    st.caption(schema[\'reference\'])\n                    st.caption(f"Reference probability: {explanation.base_values[0,predicted]:.3f}. Contributions plus reference equal {probabilities[predicted]:.3f}.")\n                except Exception:\n                    st.warning(\'The explanation is temporarily unavailable. Please treat the estimate with caution.\')\n', 'test_app.py': "\nimport numpy as np\nimport pytest\nfrom pathlib import Path\nfrom core import load_bundle, validate, estimate, explain, guidance\nfrom streamlit.testing.v1 import AppTest\n\n@pytest.fixture(scope='module')\ndef bundle():\n    return load_bundle()\n\n@pytest.mark.parametrize('value',[0,1,2,3,4])\ndef test_valid_boundaries(bundle,value):\n    model,schema=bundle\n    row, probability, category=estimate(model,schema,dict.fromkeys(schema['features'],value))\n    assert row.shape==(1,20) and category in [0,1,2]\n    assert np.isclose(probability.sum(),1) and (probability>=0).all()\n\n@pytest.mark.parametrize('value',[-1,5,1.5,None,float('nan'),float('inf'),True,'2'])\ndef test_invalid_values(bundle,value):\n    _,schema=bundle\n    with pytest.raises(ValueError): validate(dict.fromkeys(schema['features'],value),schema['features'])\n\ndef test_missing_and_extra_features(bundle):\n    _,s=bundle\n    with pytest.raises(ValueError): validate({},s['features'])\n    with pytest.raises(ValueError): validate({**dict.fromkeys(s['features'],0),'H3':5},s['features'])\n\ndef test_explanation_and_determinism(bundle):\n    m,s=bundle\n    row,p,c=estimate(m,s,dict.fromkeys(s['features'],2))\n    e=explain(m,s,row)\n    np.testing.assert_allclose(e.base_values+e.values.sum(axis=1),p[None,:],atol=1e-6)\n    np.testing.assert_allclose(m.predict_proba(row)[0],p)\n    assert all(guidance(k) for k in [0,1,2])\n\ndef app():\n    return AppTest.from_file(str(Path(__file__).parent/'app.py'),default_timeout=120).run()\n\ndef test_ui_missing_eligibility():\n    a=app()\n    assert not a.exception\n    a.button[0].click().run()\n    assert len(a.warning)>0 and not a.exception\n\ndef test_ui_missing_answers():\n    a=app()\n    a.checkbox[0].check().run()\n    a.button[0].click().run()\n    assert len(a.warning)>0 and not a.exception\n\ndef test_ui_complete_flow():\n    a=app()\n    a.checkbox[0].check().run()\n    for select in a.selectbox: select.select(0)\n    a.button[0].click().run(timeout=120)\n    assert not a.exception and len(a.warning)==0\n    assert any('intention' in s.value.lower() for s in a.subheader)\n    assert any('What influenced' in s.value for s in a.subheader)\n    assert len(a.dataframe)>=3\n", 'requirements.txt': 'numpy==2.2.6\npandas==2.2.3\nscipy==1.15.3\nscikit-learn==1.7.2\nxgboost==3.0.5\nshap==0.48.0\njoblib==1.5.1\nstreamlit==1.50.0\n', '.streamlit/config.toml': '[theme]\nprimaryColor = "#287c74"\nbackgroundColor = "#f6f8f7"\nsecondaryBackgroundColor = "#e8efed"\ntextColor = "#193b38"\nfont = "sans serif"\n[browser]\ngatherUsageStats = false\n'}
for relative, content in files.items():
    destination = APP_ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content.lstrip(), encoding='utf-8')
print('Application files generated.')

## 2. Run functional and integration tests

In [ ]:
result = subprocess.run([sys.executable,'-m','pytest','-q','test_app.py',
    '--junitxml='+str(ROOT/'outputs/app_tests.xml')],cwd=APP_ROOT,capture_output=True,text=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError('Application tests failed. Resolve failures before deployment.')
(ROOT/'outputs/app_tests.txt').write_text(result.stdout,encoding='utf-8')

## 3. Measure inference and explanation response time

In [ ]:
import importlib.util, time
import numpy as np
import pandas as pd
spec=importlib.util.spec_from_file_location('research_core',APP_ROOT/'core.py')
core=importlib.util.module_from_spec(spec); spec.loader.exec_module(core)
model,schema=core.load_bundle()
values=dict.fromkeys(schema['features'],2)
core.estimate(model,schema,values)
rows=[]
for _ in range(30):
    start=time.perf_counter()
    frame,probability,category=core.estimate(model,schema,values)
    prediction_ms=(time.perf_counter()-start)*1000
    start=time.perf_counter(); core.explain(model,schema,frame)
    rows.append({'prediction_ms':prediction_ms,'explanation_ms':(time.perf_counter()-start)*1000})
latency=pd.DataFrame(rows)
latency.to_csv(ROOT/'outputs/app_latency_runs.csv',index=False)
summary=latency.quantile([0.5,0.95]).rename(index={0.5:'p50',0.95:'p95'})
summary['total_ms']=latency.sum(axis=1).quantile([0.5,0.95]).to_numpy()
summary.to_csv(ROOT/'outputs/app_latency_summary.csv')
display(summary)
print('Colab process timings; excludes network, browser rendering and hosting cold starts.')

## 4. Create the GitHub deployment bundle
Copies only notebooks, generated application code, and selected model artifacts. Raw data and respondent-level results stay in Drive. Run the cell to download the bundle; commit it to `ChamudiDJ/Final-Year-Research`.

In [ ]:
import zipfile
bundle=ROOT/'outputs/github_bundle.zip'
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in ['notebooks','deployment']:
        for path in (ROOT/folder).rglob('*'):
            if path.is_file() and '__pycache__' not in path.parts and '.pytest_cache' not in path.parts:
                if path.suffix=='.ipynb':
                    notebook=json.loads(path.read_text(encoding='utf-8'))
                    for cell in notebook['cells']:
                        if cell['cell_type']=='code':
                            cell['outputs']=[]; cell['execution_count']=None
                    archive.writestr(str(path.relative_to(ROOT)),json.dumps(notebook,ensure_ascii=False,indent=1))
                else:
                    archive.write(path,str(path.relative_to(ROOT)))
    archive.writestr('.gitignore','data/\noutputs/\nmodels/\nreport_figures/\n__pycache__/\n.pytest_cache/\n.ipynb_checkpoints/\n.env\n.streamlit/secrets.toml\n')
from google.colab import files as colab_files
colab_files.download(str(bundle))

## 5. Host and verify
In [Streamlit Community Cloud](https://share.streamlit.io/), select `ChamudiDJ/Final-Year-Research`, the repository branch, and `deployment/app.py`; use Python 3.11. The pinned requirements are in `deployment/requirements.txt`.
After deployment, set the URL below. HTTP health is a deployment check; repeat the complete questionnaire, missing-answer, reset, and narrow-screen checks in the hosted browser. Human usability and expert evaluation are in notebook 03.

In [ ]:
APP_URL = ''  # Paste the deployed https://...streamlit.app URL.
if APP_URL:
    import requests
    from urllib.parse import urlparse
    parsed=urlparse(APP_URL)
    if parsed.scheme!='https' or not parsed.hostname or not parsed.hostname.endswith('.streamlit.app'):
        raise ValueError('Use the deployed Streamlit HTTPS URL.')
    response=requests.get(APP_URL.rstrip('/')+'/_stcore/health',timeout=60)
    response.raise_for_status()
    if response.text.strip().lower()!='ok':
        raise RuntimeError('Unexpected health response; inspect the hosted app.')
    (ROOT/'outputs/hosting_check.json').write_text(json.dumps({'url':APP_URL,'health_status':response.status_code,'health_body':response.text.strip(),'browser_acceptance':'pending'}))
    print('Hosting health passed. Complete browser acceptance testing.')
else:
    print('Hosting check pending: enter the deployed URL.')